# GM-EPC: multimodal emotion recognition in conversation on IEMOCAP

Leave-one-session-out evaluation of text-only, speech-only and fused models.
Run on Kaggle with *Save & Run All*. Inputs: the IEMOCAP release and the cached features
(`features_cache_e2v_base`). Results are written to `/kaggle/working/outputs`.

In [ ]:
# Setup: get the code and the packages needed for feature extraction
!rm -rf GM-EPC && git clone -q --depth 1 https://github.com/dujue55/GM-EPC.git
!pip install -q -r GM-EPC/requirements.txt --no-deps

import sys
sys.path.insert(0, 'GM-EPC/src')

In [ ]:
import glob
import os

import torch


def find_input(name):
    # Kaggle mounts datasets under /kaggle/input with varying parent folders
    hits = sorted(glob.glob(f'/kaggle/input/**/{name}', recursive=True))
    if not hits:
        raise FileNotFoundError(f'{name} not found under /kaggle/input')
    return hits[0]


DATA_ROOT = os.environ.get('GMEPC_DATA_ROOT') or find_input('IEMOCAP_full_release')
CACHE_DIR = os.environ.get('GMEPC_CACHE_DIR') or find_input('features_cache_e2v_base')
WORK_DIR = os.environ.get('GMEPC_WORK_DIR', '/kaggle/working')
EXTRA_CACHE_DIR = os.path.join(WORK_DIR, 'features_extra')   # features for utterances missing from the cache
OUTPUT_DIR = os.path.join(WORK_DIR, 'outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)

SEEDS = [0, 1, 2]
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'data: {DATA_ROOT}\ncache: {CACHE_DIR}\ndevice: {DEVICE}')

## 1. Data and cache check

Load the corpus, check the utterance order, and extract features for any utterance the cache
does not contain.

In [ ]:
from collections import Counter

from dataset import EMO_MAP, FeatureStore, audit_cache, load_corpus

corpus, dropped = load_corpus(DATA_ROOT)
utterances = [u for entry in corpus.values() for u in entry['utterances']]
print(f'{len(corpus)} dialogues, {len(utterances)} utterances, {len(dropped)} without transcript')
print('Target labels:', Counter(u['label'] for u in utterances if u['label'] in EMO_MAP))

# Order check: speakers should alternate in time within a dialogue
example = sorted(corpus)[0]
for u in corpus[example]['utterances'][:6]:
    print(f"  {u['start']:8.2f}s  {u['utt_id']}  {u['label']}")

report = audit_cache(corpus, dropped, FeatureStore([CACHE_DIR], 'e2v'))
print(f"\nMissing from the cache: {len(report['missing'])}")
print('Labels of missing utterances:', Counter(label for _, label in report['missing']))

In [ ]:
from features import extract_features, save_cache

missing = {utt_id for utt_id, _ in report['missing']}
for session in sorted({corpus[u.rsplit('_', 1)[0]]['session'] for u in missing}):
    todo = [u for u in utterances if u['utt_id'] in missing and corpus[u['utt_id'].rsplit('_', 1)[0]]['session'] == session]
    feats = extract_features(todo, DEVICE)
    save_cache(EXTRA_CACHE_DIR, session, [u['utt_id'] for u in todo], feats)
    print(f'{session}: extracted {len(todo)} utterances')

CACHE_DIRS = [CACHE_DIR, EXTRA_CACHE_DIR]
stores = {tag: FeatureStore(CACHE_DIRS, tag) for tag in ('e2v', 'wavlm')}
for tag, store in stores.items():
    assert not audit_cache(corpus, dropped, store)['missing'], f'{tag}: features still missing'
print('All utterances have features.')

## 2. Timing check (one fold)

In [ ]:
import time

from trainer import DEFAULT_CONFIG, SETTINGS, run_fold

cfg = {**DEFAULT_CONFIG, **SETTINGS['Gated Fusion (E2V)']}
start = time.time()
summary, _ = run_fold(corpus, stores['e2v'], 'Session1', cfg, seed=0, device=DEVICE)
fold_time = time.time() - start
print({k: summary[k] for k in ['n_train', 'n_val', 'n_test', 'best_epoch', 'test_macro_f1']})
print(f'{fold_time:.0f}s for one fold -> about {fold_time * 5 * len(SEEDS) * len(SETTINGS) / 60:.0f} min in total')

## 3. All settings

8 settings x 5 folds x 3 seeds. Results are saved after every setting, and settings that
are already in the saved files are skipped, so an interrupted run can be resumed.

In [ ]:
import pandas as pd

from trainer import run_setting

folds_path = os.path.join(OUTPUT_DIR, 'folds.csv')
preds_path = os.path.join(OUTPUT_DIR, 'predictions.csv')
folds = pd.read_csv(folds_path) if os.path.exists(folds_path) else pd.DataFrame()
preds = pd.read_csv(preds_path) if os.path.exists(preds_path) else pd.DataFrame()

for name in SETTINGS:
    if len(folds) and name in set(folds['setting']):
        print(f'{name}: already done')
        continue
    f, p = run_setting(name, corpus, stores, SEEDS, DEVICE)
    folds, preds = pd.concat([folds, f], ignore_index=True), pd.concat([preds, p], ignore_index=True)
    folds.to_csv(folds_path, index=False)
    preds.to_csv(preds_path, index=False)

## 4. Results

Each seed is averaged over the five folds first; the table shows mean and standard
deviation across the three seeds.

In [ ]:
from trainer import summarize

summary = summarize(folds)
summary.to_csv(os.path.join(OUTPUT_DIR, 'summary.csv'))
summary.round(4)

In [ ]:
# Per-fold macro-F1, averaged over seeds
folds.pivot_table(index='setting', columns='test_session', values='test_macro_f1').reindex(list(SETTINGS)).round(4)

## 5. Figures

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix

from dataset import CLASS_NAMES

FIG_DIR = os.path.join(OUTPUT_DIR, 'figures')
os.makedirs(FIG_DIR, exist_ok=True)

# Macro-F1 per setting (mean and std across seeds)
fig, ax = plt.subplots(figsize=(7, 3.6))
y = np.arange(len(summary))[::-1]
ax.barh(y, summary['test_macro_f1_mean'], xerr=summary['test_macro_f1_std'], color='#2a78d6', capsize=3)
ax.set_yticks(y)
ax.set_yticklabels(summary.index)
ax.set_xlabel('Test macro-F1')
lo = (summary['test_macro_f1_mean'] - summary['test_macro_f1_std'].fillna(0)).min()
ax.set_xlim(max(0, np.floor(lo * 10) / 10 - 0.1), 1)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'macro_f1.png'), dpi=200)
plt.show()

In [ ]:
# Confusion matrix of gated fusion (E2V), all folds and seeds pooled, rows normalised
p = preds[preds['setting'] == 'Gated Fusion (E2V)']
cm = confusion_matrix(p['label'], p['pred'], normalize='true') * 100
fig, ax = plt.subplots(figsize=(4, 3.6))
ax.imshow(cm, cmap='Blues', vmin=0, vmax=100)
for i in range(4):
    for j in range(4):
        ax.text(j, i, f'{cm[i, j]:.1f}', ha='center', va='center', color='white' if cm[i, j] > 60 else 'black')
ax.set_xticks(range(4))
ax.set_xticklabels(CLASS_NAMES)
ax.set_yticks(range(4))
ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'confusion_gated_e2v.png'), dpi=200)
plt.show()

In [ ]:
# Text weight of the gate at the target utterance, by true emotion (gated fusion, E2V)
fig, ax = plt.subplots(figsize=(4.5, 3.2))
ax.boxplot([p.loc[p['label'] == k, 'gate_target'] for k in range(4)], showfliers=False)
ax.set_xticks(range(1, 5))
ax.set_xticklabels(CLASS_NAMES)
ax.axhline(0.5, color='grey', linestyle=':', linewidth=1)
ax.set_ylabel('Gate (text weight)')
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'gate_by_emotion.png'), dpi=200)
plt.show()
p.groupby('label')[['gate_target', 'gate_dim_std', 'gate_time_std']].describe().round(3)